# English -> Ekegusii Translation (Skeleton-Masking + Gemini) - FULL RUN

**`TEST_MODE` is now OFF** - this runs all **6,781 unique skeletons**, covering all 15,000 rows, in one go. Real computed cost: **~$1.43** (well within your $10 balance).

**Confirmed working** from your earlier test batch - 0 failed/blank, contamination check showed genuinely Ekegusii output (not Swahili-leaking).

**This can take a while** - 6,781 calls at your MAX_WORKERS=6 parallelism. Keep the Colab tab open and active until it finishes; everything is in local session storage, so a disconnect would lose progress.

**Before running:** upload `df_with_swahili.csv` and `kenyan_psa_synthetic_15000.csv`.

## Step 1: Install dependencies

In [1]:
!pip install -q google-genai pandas tqdm


## Step 2: Upload your two input data files

In [2]:
from google.colab import files
print('Upload df_with_swahili.csv and kenyan_psa_synthetic_15000.csv:')
uploaded = files.upload()


Upload df_with_swahili.csv and kenyan_psa_synthetic_15000.csv:


Saving df_with_swahili.csv to df_with_swahili.csv
Saving kenyan_psa_synthetic_15000.csv to kenyan_psa_synthetic_15000.csv


In [8]:
import os
print(os.path.exists('ekegusii_api_skeleton_checkpoint.csv'))

True


In [ ]:
"""
English -> Ekegusii translation using Google's Gemini API, with
slot-masking to cut cost/calls and explicit ID-based merging to prevent
row mismatches.

ADAPTED FROM THE OPENAI VERSION - same masking logic, same checkpointing,
same quality-filtered few-shot selection, same contamination check. Only
the actual API client/call changed, since OpenAI billing kept rejecting
cards across multiple countries.

Uses the current google-genai SDK (NOT the deprecated google.generativeai
package). Model defaults to "gemini-3.1-flash-lite" - real computed cost
for this exact task: ~$1.43 for the full 6,781-skeleton run (verified
against actual current Gemini pricing).

BEFORE YOU RUN THIS
--------------------
1. Get a Gemini API key from https://aistudio.google.com/apikey
   (free to generate - no card needed for this step).
2. To actually use a paid tier (needed for this volume - free tier is
   only ~20 requests/day), link Google Cloud billing at
   https://console.cloud.google.com/billing to the same project your
   key belongs to.
3. Install the SDK:
       pip install google-genai
4. Put this script in the same folder as:
       - df_with_swahili.csv                  (few-shot source)
       - kenyan_psa_synthetic_15000.csv        (English-only data to translate)
5. Run it - you'll be prompted for your Gemini API key.

RUN IN COLAB
1. Upload df_with_swahili.csv and kenyan_psa_synthetic_15000.csv
2. Run this cell - you'll be prompted for your Gemini API key
3. Resumable via checkpoint if it stops partway
"""

import csv
import hashlib
import json
import os
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass

import pandas as pd
from google import genai
from tqdm import tqdm
csv.field_size_limit(2**31 - 1)

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
# Everything - inputs AND outputs - now lives in Colab's local session
# storage (no Google Drive needed). Simpler, but a real tradeoff worth
# knowing: if this Colab session disconnects (auto-disconnects after
# ~90 min idle, or if you close the tab) BEFORE you've finished - e.g.
# you run out of API credits and need to top up first - the checkpoint
# and any partial output here would be LOST, and you'd re-pay for
# already-translated skeletons on the next run. Keep this tab open and
# active until the run finishes, or top up credits quickly if you hit
# insufficient_quota, rather than walking away for a long time.
REFERENCE_CSV = "df_with_swahili.csv"           # uploaded via files.upload() - few-shot source
SYNTHETIC_CSV = "kenyan_psa_synthetic_15000.csv"  # uploaded via files.upload() - <-- EDIT this filename if yours differs

CHECKPOINT_CSV = "ekegusii_api_skeleton_checkpoint.csv"
FINAL_OUTPUT_CSV = "kenyan_psa_15000_ekegusii_api.csv"

MODEL = "gemini-3.1-flash-lite"    # confirmed real pricing: $0.25/MTok input, $1.50/MTok output - ~$1.43 for this full task
N_FEWSHOT_EXAMPLES = 8
MAX_WORKERS = 4
MAX_RETRIES = 3
RANDOM_SEED = 42          # fixed seed - reproducible few-shot sampling across runs

TEST_MODE = False         # full run - all 6,781 unique skeletons, covering all 15,000 rows
TEST_SKELETON_LIMIT = 20  # only this many unique skeletons while TEST_MODE is True

random.seed(RANDOM_SEED)

for required_file, label in [(REFERENCE_CSV, "reference/few-shot file"), (SYNTHETIC_CSV, "synthetic data file")]:
    if not os.path.exists(required_file):
        raise SystemExit(
            f"Could not find the {label} at: {required_file}\n"
            f"Run the 'Upload your two input data files' cell above and select it, "
            f"or check the filename matches exactly (including capitalization)."
        )

api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)


# ------------------------------------------------------------------
# STEP 1: mask slots -> unique skeletons
# ------------------------------------------------------------------
KENYA_COUNTIES = [
    "Mombasa","Kwale","Kilifi","Tana River","Lamu","Taita-Taveta","Taita Taveta","Garissa","Wajir","Mandera",
    "Marsabit","Isiolo","Meru","Tharaka-Nithi","Tharaka Nithi","Embu","Kitui","Machakos","Makueni","Nyandarua",
    "Nyeri","Kirinyaga","Murang'a","Muranga","Kiambu","Turkana","West Pokot","Samburu","Trans Nzoia","Uasin Gishu",
    "Elgeyo-Marakwet","Elgeyo Marakwet","Nandi","Baringo","Laikipia","Nakuru","Narok","Kajiado","Kericho","Bomet",
    "Kakamega","Vihiga","Bungoma","Busia","Siaya","Kisumu","Homa Bay","Migori","Kisii","Nyamira","Nairobi",
]
KENYA_COUNTIES_SORTED = sorted(set(KENYA_COUNTIES), key=len, reverse=True)
MONTHS = ["January","February","March","April","May","June","July","August",
          "September","October","November","December"]
MONTH_TO_NUM = {m: i + 1 for i, m in enumerate(MONTHS)}


def mask_text(text):
    slots = {}
    counters = {"COUNTY": 0, "MONTH": 0, "YEAR": 0, "NUM": 0}
    masked = text

    for county in KENYA_COUNTIES_SORTED:
        pattern = re.compile(r'\b' + re.escape(county) + r'\b')
        while pattern.search(masked):
            key = f"§COUNTY{counters['COUNTY']}§"
            slots[key] = county
            masked = pattern.sub(key, masked, count=1)
            counters['COUNTY'] += 1

    for month in MONTHS:
        pattern = re.compile(r'\b' + re.escape(month) + r'\b')
        while pattern.search(masked):
            key = f"§MONTH{counters['MONTH']}§"
            slots[key] = month
            masked = pattern.sub(key, masked, count=1)
            counters['MONTH'] += 1

    year_pattern = re.compile(r'\b(19|20)\d{2}\b')
    while year_pattern.search(masked):
        m = year_pattern.search(masked)
        key = f"§YEAR{counters['YEAR']}§"
        slots[key] = m.group(0)
        masked = masked[:m.start()] + key + masked[m.end():]
        counters['YEAR'] += 1

    num_pattern = re.compile(r'\b\d+(\.\d+)?%?\b')
    while num_pattern.search(masked):
        m = num_pattern.search(masked)
        key = f"§NUM{counters['NUM']}§"
        slots[key] = m.group(0)
        masked = masked[:m.start()] + key + masked[m.end():]
        counters['NUM'] += 1

    return masked, slots


def substitute_slots(text, slots):
    out = text
    for key, value in slots.items():
        if key.startswith("§COUNTY"):
            out = out.replace(key, value)
        elif key.startswith("§MONTH"):
            # UNVERIFIED - confirm with a native Ekegusii speaker that this
            # month-as-number convention is genuinely correct before trusting it.
            out = out.replace(key, f"omotienyi {MONTH_TO_NUM.get(value, value)}")
        else:
            out = out.replace(key, value)
    return out


df = pd.read_csv(SYNTHETIC_CSV, encoding="utf-8-sig")  # utf-8-sig: guards against a hidden BOM character
print(f"Loaded {len(df)} rows")

required_cols = ["PSA_ID", "Domain", "English"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise SystemExit(f"Missing column(s) {missing} in {SYNTHETIC_CSV}. Columns present: {list(df.columns)}")

skeleton_of_row = {}
slots_of_row = {}
skeleton_text_of_id = {}

for _, row in df.iterrows():
    pid = row["PSA_ID"]
    skeleton_text, slots = mask_text(row["English"])
    skel_id = "SK" + hashlib.sha1(skeleton_text.encode("utf-8")).hexdigest()[:10]
    skeleton_of_row[pid] = skel_id
    slots_of_row[pid] = slots
    skeleton_text_of_id[skel_id] = skeleton_text

unique_skeleton_ids = list(skeleton_text_of_id.keys())
print(f"Unique skeletons to translate: {len(unique_skeleton_ids)} "
      f"({100 * (1 - len(unique_skeleton_ids)/len(df)):.1f}% reduction)")

if TEST_MODE:
    unique_skeleton_ids = unique_skeleton_ids[:TEST_SKELETON_LIMIT]
    print(f"TEST_MODE is on: only translating the first {len(unique_skeleton_ids)} unique skeletons. "
          f"Set TEST_MODE = False once you've reviewed these.")


# ------------------------------------------------------------------
# STEP 2: build few-shot context from df_with_swahili.csv, quality-
# filtered on len_ratio, then translate each unique skeleton.
# ------------------------------------------------------------------
def load_reference_pairs(path):
    rows = []
    with open(path, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            eng = r.get("en_clean", "").strip()
            guz = r.get("guz_clean", "").strip()
            len_ratio_raw = r.get("len_ratio", "")
            try:
                len_ratio = float(len_ratio_raw)
            except (ValueError, TypeError):
                len_ratio = None
            if eng and guz:
                rows.append((eng, guz, len_ratio))

    ratios = [r[2] for r in rows if r[2] is not None]
    if ratios:
        ratios_sorted = sorted(ratios)
        q1 = ratios_sorted[len(ratios_sorted) // 4]
        q3 = ratios_sorted[3 * len(ratios_sorted) // 4]
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        before = len(rows)
        rows = [r for r in rows if r[2] is None or (lower <= r[2] <= upper)]
        print(f"  len_ratio quality filter: kept {len(rows)}/{before} rows "
              f"(bounds [{lower:.3f}, {upper:.3f}])")

    return [(eng, guz) for eng, guz, _ in rows]


reference_pairs = load_reference_pairs(REFERENCE_CSV)
print(f"Loaded {len(reference_pairs)} quality-filtered reference pairs for few-shot context")


def build_prompt(skeleton_text, fewshot):
    examples_block = "\n".join(
        f"English: {en}\nEkegusii: {guz}" for en, guz in fewshot
    )
    return f"""You are translating short Kenyan public service announcements from
English to Ekegusii (Gusii), a low-resource Bantu language spoken in Kenya.

Here are real English-Ekegusii example pairs to guide style and vocabulary:

{examples_block}

Now translate the following text to Ekegusii. It may contain placeholder
tokens like §COUNTY0§, §MONTH0§, §YEAR0§, §NUM0§ - copy these placeholder
tokens through UNCHANGED in your output, in the same position, do not
translate or alter them. Only translate the surrounding English words.

Text to translate:
{skeleton_text}

Respond with ONLY the Ekegusii translation, nothing else - no explanation,
no quotes, no restatement of the English."""


translated_ekegusii = {}

if os.path.exists(CHECKPOINT_CSV):
    with open(CHECKPOINT_CSV, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            if r["ekegusii_skeleton"].strip():
                translated_ekegusii[r["skeleton_id"]] = r["ekegusii_skeleton"]
    print(f"Resumed {len(translated_ekegusii)} already-translated skeletons from checkpoint")

todo_ids = [sid for sid in unique_skeleton_ids if sid not in translated_ekegusii]
print(f"Still need to translate: {len(todo_ids)} skeletons")


def random_fewshot(n):
    return random.sample(reference_pairs, min(n, len(reference_pairs)))


def translate_one(skel_id):
    text = skeleton_text_of_id[skel_id]
    fewshot = random_fewshot(N_FEWSHOT_EXAMPLES)
    prompt = build_prompt(text, fewshot)
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=genai.types.GenerateContentConfig(temperature=0.2),
            )
            result = resp.text.strip() if resp.text else ""
            if result:
                return skel_id, result
        except Exception as e:
            print(f"  (retrying {skel_id}, attempt {attempt+1}/{MAX_RETRIES}: {e})")
            time.sleep(2 + attempt * 2)
    return skel_id, ""


def save_checkpoint():
    with open(CHECKPOINT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["skeleton_id", "ekegusii_skeleton"])
        for sid, text in translated_ekegusii.items():
            writer.writerow([sid, text])


CHECKPOINT_EVERY = 250
processed_since_save = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(translate_one, sid): sid for sid in todo_ids}
    for future in tqdm(as_completed(futures), total=len(futures)):
        skel_id, result = future.result()
        translated_ekegusii[skel_id] = result
        processed_since_save += 1
        if processed_since_save >= CHECKPOINT_EVERY:
            save_checkpoint()
            processed_since_save = 0

save_checkpoint()
failed = [sid for sid, text in translated_ekegusii.items() if not text.strip()]
print(f"Done translating skeletons. Failed/blank: {len(failed)} (re-run this cell to retry just those)")


# ------------------------------------------------------------------
# STEP 2.5: Swahili-vs-Ekegusii contamination check
# ------------------------------------------------------------------
# Ekegusii is closely related to Kiswahili, and the model has almost
# certainly seen far more Kiswahili than Ekegusii text during training.
# This means a real risk: fluent-sounding output that's actually closer
# to Kiswahili than genuine Ekegusii. This does NOT confirm a
# translation is correct - only a native Ekegusii speaker can do that -
# it flags output worth extra scrutiny.
def tokenize_words(text):
    return set(re.findall(r"[a-zA-Z']+", str(text).lower()))


def build_vocab_sets(parallel_path):
    ekegusii_vocab, swahili_vocab = set(), set()
    with open(parallel_path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            guz = row.get("guz_clean", "")
            sw = row.get("sw_clean", "")
            if guz:
                ekegusii_vocab |= tokenize_words(guz)
            if sw:
                swahili_vocab |= tokenize_words(sw)
    return ekegusii_vocab, swahili_vocab


def contamination_score(text, ekegusii_vocab, swahili_vocab):
    tokens = tokenize_words(text)
    if not tokens:
        return 0.0
    overlap_sw = len(tokens & swahili_vocab) / len(tokens)
    overlap_guz = len(tokens & ekegusii_vocab) / len(tokens)
    return overlap_sw - overlap_guz


CONTAMINATION_FLAG_THRESHOLD = 0.15

print("\nBuilding Ekegusii/Swahili reference vocabularies for contamination check...")
ekegusii_vocab, swahili_vocab = build_vocab_sets(REFERENCE_CSV)
print(f"  -> {len(ekegusii_vocab)} Ekegusii words, {len(swahili_vocab)} Swahili words in reference vocab.")


# ------------------------------------------------------------------
# STEP 3: substitute slots back in, keyed by PSA_ID -> skeleton_id ->
# translation. No positional merging anywhere.
# ------------------------------------------------------------------
output_rows = []
for _, row in df.iterrows():
    pid = row["PSA_ID"]
    skel_id = skeleton_of_row[pid]
    ekeg_skeleton = translated_ekegusii.get(skel_id, "")
    ekeg_full = substitute_slots(ekeg_skeleton, slots_of_row[pid]) if ekeg_skeleton else ""
    score = contamination_score(ekeg_full, ekegusii_vocab, swahili_vocab) if ekeg_full else 0.0
    output_rows.append({
        "PSA_ID": pid,
        "Domain": row["Domain"],
        "English": row["English"],
        "Ekegusii": ekeg_full,
        "contamination_score": round(score, 3),
        "flagged": score > CONTAMINATION_FLAG_THRESHOLD,
    })

out_df = pd.DataFrame(output_rows)
out_df.to_csv(FINAL_OUTPUT_CSV, index=False)

filled = out_df["Ekegusii"].str.strip().astype(bool).sum()
print(f"\nSaved {len(out_df)} rows to {FINAL_OUTPUT_CSV}")
print(f"Ekegusii filled: {filled}/{len(out_df)}")

translated_rows = out_df[out_df["Ekegusii"].str.strip().astype(bool)]
flagged_count = translated_rows["flagged"].sum()
print(f"Contamination check: {flagged_count}/{len(translated_rows)} translated rows "
      f"look more Swahili than Ekegusii (score > {CONTAMINATION_FLAG_THRESHOLD}).")
print("This is a coarse heuristic, NOT a correctness check - a native Ekegusii speaker "
      "should still review a sample of the UNFLAGGED rows too, not just the flagged ones.")

print("\nSpot check (confirm English and Ekegusii topics visibly match):")
print(translated_rows[["PSA_ID", "English", "Ekegusii", "contamination_score"]].sample(
    min(5, filled), random_state=1).to_string())


Enter your Gemini API key: ··········
Loaded 15000 rows
  (retrying SK9617bea27e, attempt 1/3: Server disconnected without sending a response.)
  (retrying SK99df5d9005, attempt 1/3: Server disconnected without sending a response.)
  (retrying SKd4a5f9e204, attempt 1/3: Server disconnected without sending a response.)
Unique skeletons to translate: 6781 (54.8% reduction)
  len_ratio quality filter: kept 4719/4816 rows (bounds [0.183, 1.337])
Loaded 4719 quality-filtered reference pairs for few-shot context
Resumed 2831 already-translated skeletons from checkpoint
Still need to translate: 3950 skeletons


  4%|▍         | 149/3950 [00:45<35:31,  1.78it/s]

  (retrying SK9e6dcb9675, attempt 1/3: Server disconnected without sending a response.)
  (retrying SK26ad8dcf44, attempt 1/3: Server disconnected without sending a response.)
  (retrying SK35a965ab25, attempt 1/3: [SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:2580))
  (retrying SKb2c36926b6, attempt 1/3: [SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2580))
  (retrying SKcced3e69dd, attempt 1/3: [SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:2580))
  (retrying SK67fc8ce9c8, attempt 1/3: [SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2580))


  4%|▍         | 152/3950 [02:31<16:48:28, 15.93s/it]

  (retrying SKb50b40b132, attempt 1/3: [SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:2580))
  (retrying SK3450c396cd, attempt 1/3: Server disconnected without sending a response.)


...

  (retrying SK7cc5401e87, attempt 3/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SKc5fe605440, attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SK68ee83e7d3, attempt 2/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retr

 14%|█▍        | 556/3950 [11:17<2:40:17,  2.83s/it]

  (retrying SKa76b0d12a1, attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})


 14%|█▍        | 557/3950 [11:19<2:17:31,  2.43s/it]

  (retrying SKceed3b7a56, attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SK423dfd61ea, attempt 2/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})


In [9]:
from google.colab import files
files.download('ekegusii_api_skeleton_checkpoint.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  (retrying SK1fed44b377, attempt 3/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SKcf48b2bec7, attempt 3/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SK10505c110a, attempt 3/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retr

## Step 3: The full translation run

You'll be prompted for your Gemini API key.

Enter your Gemini API key: ··········
Loaded 15000 rows
Unique skeletons to translate: 6781 (54.8% reduction)
  len_ratio quality filter: kept 4719/4816 rows (bounds [0.183, 1.337])
Loaded 4719 quality-filtered reference pairs for few-shot context
Still need to translate: 6781 skeletons


 36%|███▌      | 2437/6781 [43:24<1:01:47,  1.17it/s]

  (retrying SK0b715af8f7, attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SK0b715af8f7, attempt 2/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SK0b715af8f7, attempt 3/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})


 36%|███▌      | 2438/6781 [43:37<5:07:41,  4.25s/it]

  (retrying SK9ceba8f99e, attempt 1/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SK9ceba8f99e, attempt 2/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})
  (retrying SK9ceba8f99e, attempt 3/3: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}})


...

 49%|████▉     | 3349/6781 [1:09:51<11:44,  4.87it/s]

  (retrying SK2c8cb1fe6c, attempt 1/3: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})


 52%|█████▏    | 3499/6781 [1:12:14<1:07:45,  1.24s/it]


  (retrying SK47455dbbbe, attempt 1/3: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})


KeyboardInterrupt: 

In [4]:
import os
print(os.path.exists('ekegusii_api_skeleton_checkpoint.csv'))

True


## Step 4: Check flagged rows (contamination check)

In [ ]:
import pandas as pd
result = pd.read_csv('kenyan_psa_15000_ekegusii_api.csv')
flagged = result[result['flagged'] == True]
print(f'Total rows: {len(result)}')
print(f'Ekegusii filled: {result["Ekegusii"].str.strip().astype(bool).sum()}')
print(f'{len(flagged)} rows flagged as possibly more Swahili than Ekegusii')


## Step 5: Download the final translated file

In [ ]:
from google.colab import files
files.download('kenyan_psa_15000_ekegusii_api.csv')
